## Data prep
Keep the useful columns and remove missing and impossible values.

In [9]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

df = pd.read_csv("data/athletes.csv")

# remove rows with missing key values, then columns that are not relevant
df = df.dropna(subset=["region", "age", "weight", "height", "howlong", "gender", "eat",
                       "train", "background", "experience", "schedule",
                       "deadlift", "candj", "snatch", "backsq"])
df = df.drop(columns=["affiliate", "team", "name", "athlete_id", "fran", "helen", "grace",
                      "filthy50", "fgonebad", "run400", "run5k", "pullups", "train"])

# remove outliers
df = df[df["weight"] < 1500]
df = df[df["gender"] != "--"]
df = df[df["age"] >= 18]
df = df[(df["height"] < 96) & (df["height"] > 48)]
df = df[((df["deadlift"] > 0) & (df["deadlift"] <= 1105))
        | ((df["gender"] == "Female") & (df["deadlift"] <= 636))]
df = df[(df["candj"] > 0) & (df["candj"] <= 395)]
df = df[(df["snatch"] > 0) & (df["snatch"] <= 496)]
df = df[(df["backsq"] > 0) & (df["backsq"] <= 1069)]

# clean survey data
df = df.replace({"Decline to answer|": np.nan})
df = df.dropna(subset=["background", "experience", "schedule", "howlong", "eat"])

df.to_csv("data/athletes_clean.csv", index=False)
df.shape

(30029, 14)

## Start H2O and load data
Point H2O at the conda Java, start the server, and split the data 80/20 into train and validation.

In [10]:
import os, sys
import h2o
from h2o.automl import H2OAutoML

# let H2O find the Java installed in this conda environment
os.environ["PATH"] = os.path.join(sys.prefix, "Library", "bin") + os.pathsep + os.environ["PATH"]
os.environ["JAVA_HOME"] = os.path.join(sys.prefix, "Library")

h2o.init()
data = h2o.import_file("data/athletes_clean.csv")
train, valid = data.split_frame(ratios=[0.8], seed=42)
target = "deadlift"

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,30 mins 35 secs
H2O_cluster_timezone:,America/New_York
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,2 months and 20 days
H2O_cluster_name:,H2O_from_python_allis_a44shc
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,5.847 Gb
H2O_cluster_total_cores:,16
H2O_cluster_allowed_cores:,16
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


## AutoML helper
Runs AutoML (up to 15 models, no Deep Learning) on a feature list and prints the top 3 models by validation score.

In [11]:
import time


def run(features):
    # removing deep learning since it requires GPU access and slow on CPU
    aml = H2OAutoML(max_models=15, seed=42, exclude_algos=["DeepLearning"])
    start = time.time()
    aml.train(x=features, y=target, training_frame=train, validation_frame=valid)
    aml.total_secs = time.time() - start  # total wall-clock time of the AutoML run
    print(f"Total AutoML run time: {aml.total_secs:.1f} s")
    board = aml.leaderboard.as_data_frame()
    print("Top 3 by validation score")
    print(board[["model_id", "rmse"]])
    return aml

## All features

In [12]:
features = [c for c in data.columns if c != target]
aml = run(features)

AutoML progress: |█
01:45:58.701: User specified a validation frame with cross-validation still enabled. Please note that the models will still be validated using cross-validation only, the validation frame will be used to provide purely informative validation metrics on the trained models.
01:45:58.702: AutoML: XGBoost is not available; skipping it.

██████████████████████████████████████████████████████████████| (done) 100%
Total AutoML run time: 110.5 s
Top 3 by validation score
                                                  model_id       rmse
0      StackedEnsemble_AllModels_1_AutoML_6_20261101_14558  36.601986
1   StackedEnsemble_BestOfFamily_1_AutoML_6_20261101_14558  36.633465
2                            GBM_5_AutoML_6_20261101_14558  37.003890
3                            GLM_1_AutoML_6_20261101_14558  37.025570
4                            GBM_2_AutoML_6_20261101_14558  37.092421
5               GBM_grid_1_AutoML_6_20261101_14558_model_7  37.169962
6               GBM_gri

## Data insights from the AutoML run
What AutoML reports about the data and the run: the validation checks it made, how long it took, and every model it built.

- **Data:** 30,029 rows and 14 columns after cleaning, split 80/20 into train and validation.
- **Validation:** AutoML used 5-fold cross-validation. The validation frame was only used for extra, informational metrics, so the leaderboard RMSE is a cross-validation score.
- **Skipped:** XGBoost was not available on this machine and was skipped. Deep Learning was excluded on purpose.
- **Run:** it built 17 models in 59 seconds: 12 GBM, 1 GLM, 1 XRT, 1 DRF and 2 Stacked Ensembles.
- **Results:** RMSE ranged from 36.6 (best Stacked Ensemble) to 40.9 (worst GBM grid model). The best single model was `GBM_5` at 37.0. The ensembles gained only about 0.4 lbs of RMSE over it.

In [13]:
pd.set_option("display.max_colwidth", None)

print(aml.training_info)

# what AutoML checked and decided before training
log = aml.event_log.as_data_frame()
print(log[log["stage"] == "Validation"][["level", "message"]].to_string(index=False))

# every model it built
aml.leaderboard.as_data_frame()

{'creation_epoch': '1793511959', 'start_epoch': '1793511959', 'start_GLM_def_1': '1793511959', 'start_GBM_def_5': '1793511959', 'start_DRF_def_1': '1793511961', 'start_GBM_def_2': '1793511975', 'start_GBM_def_3': '1793511976', 'start_GBM_def_4': '1793511977', 'start_DRF_XRT': '1793511979', 'start_GBM_def_1': '1793511995', 'start_GBM_grid_1': '1793511997', 'start_StackedEnsemble_best_of_family_xglm': '1793512013', 'start_StackedEnsemble_all_xglm': '1793512013', 'stop_epoch': '1793512015', 'duration_secs': '56'}
level                                                                                                                                                                                                                                                           message
 INFO                                                                                                                                                                                                                       

,model_id,rmse,mse,mae,rmsle,mean_residual_deviance
0,StackedEnsemble_AllModels_1_AutoML_6_20261101_14558,36.601986,1339.705346,27.384316,0.178142,1339.705346
1,StackedEnsemble_BestOfFamily_1_AutoML_6_20261101_14558,36.633465,1342.010792,27.409302,0.177582,1342.010792
2,GBM_5_AutoML_6_20261101_14558,37.003890,1369.287867,27.674221,0.176320,1369.287867
3,GLM_1_AutoML_6_20261101_14558,37.025570,1370.892845,27.663852,0.181688,1370.892845
4,GBM_2_AutoML_6_20261101_14558,37.092421,1375.847678,27.801779,0.177029,1375.847678
5,GBM_grid_1_AutoML_6_20261101_14558_model_7,37.169962,1381.606075,27.815125,0.185251,1381.606075
6,GBM_grid_1_AutoML_6_20261101_14558_model_1,37.182433,1382.533299,27.834077,0.176768,1382.533299
7,XRT_1_AutoML_6_20261101_14558,37.235194,1386.459663,27.939543,0.175452,1386.459663
8,GBM_grid_1_AutoML_6_20261101_14558_model_6,37.301697,1391.416602,27.928611,0.183544,1391.416602
9,GBM_3_AutoML_6_20261101_14558,37.330805,1393.589018,28.040721,0.177128,1393.589018


## Top 5 features (from the best model)
The best model is a Stacked Ensemble, which has no feature importance. The importance comes from the best-ranked single model, `GBM_5`.

1. `backsq` (70.1%)
2. `candj` (21.0%)
3. `snatch` (2.7%)
4. `weight` (1.5%)
5. `schedule` (1.1%)

`backsq` and `candj` carry about 91% of the importance.

In [14]:
print("Best model:", aml.leader.model_id)

# a Stacked Ensemble has no feature importance, so use the best model that does
for model_id in aml.leaderboard.as_data_frame()["model_id"]:
    imp = h2o.get_model(model_id).varimp(use_pandas=True)
    if imp is not None:
        print("Feature importance from:", model_id)
        break

imp.head(5)[["variable", "percentage"]]

Best model: StackedEnsemble_AllModels_1_AutoML_6_20261101_14558
Feature importance from: GBM_5_AutoML_6_20261101_14558


,variable,percentage
0,backsq,0.700712
1,candj,0.210493
2,snatch,0.026725
3,weight,0.014672
4,schedule,0.011173


## Top 3 features only
AutoML was rerun on `backsq`, `candj` and `snatch`, the top 3 features from the run above.

In [15]:
top3 = imp["variable"].head(3).tolist()
print(top3)
aml3 = run(top3)

['backsq', 'candj', 'snatch']
AutoML progress: |█
01:48:07.490: User specified a validation frame with cross-validation still enabled. Please note that the models will still be validated using cross-validation only, the validation frame will be used to provide purely informative validation metrics on the trained models.
01:48:07.490: AutoML: XGBoost is not available; skipping it.

██████████████████████████████████████████████████████████████| (done) 100%
Total AutoML run time: 72.8 s
Top 3 by validation score
                                                  model_id       rmse
0      StackedEnsemble_AllModels_1_AutoML_7_20261101_14807  37.993438
1   StackedEnsemble_BestOfFamily_1_AutoML_7_20261101_14807  38.010017
2                            GBM_5_AutoML_7_20261101_14807  38.052683
3                            GBM_2_AutoML_7_20261101_14807  38.101769
4                            GBM_3_AutoML_7_20261101_14807  38.126797
5                            GBM_1_AutoML_7_20261101_14807  38.2

## All features vs top 3 features
The top 3 models by validation score (cross-validation RMSE, in lbs) from each run, side by side.

| Rank | All features | Top 3 features |
|---|---|---|
| 1 | Stacked Ensemble AllModels, 36.60 | Stacked Ensemble AllModels, 37.99 |
| 2 | Stacked Ensemble BestOfFamily, 36.63 | Stacked Ensemble BestOfFamily, 38.01 |
| 3 | GBM_5, 37.00 | GBM_5, 38.05 |

The same three models win both times. Using only 3 of 13 features raises the best RMSE by about 1.4 lbs (3.8%), so most of the predictive power is in `backsq`, `candj` and `snatch`.

In [16]:
def top3_table(a):
    return a.leaderboard.as_data_frame().head(3)[["model_id", "rmse"]]


pd.concat({"All features": top3_table(aml), "Top 3 features": top3_table(aml3)})

model_id  \
All features   0     StackedEnsemble_AllModels_1_AutoML_6_20261101_14558   
               1  StackedEnsemble_BestOfFamily_1_AutoML_6_20261101_14558   
               2                           GBM_5_AutoML_6_20261101_14558   
Top 3 features 0     StackedEnsemble_AllModels_1_AutoML_7_20261101_14807   
               1  StackedEnsemble_BestOfFamily_1_AutoML_7_20261101_14807   
               2                           GBM_5_AutoML_7_20261101_14807   

                       rmse  
All features   0  36.601986  
               1  36.633465  
               2  37.003890  
Top 3 features 0  37.993438  
               1  38.010017  
               2  38.052683

## Time and speed
Total run time of each AutoML run, and the training time and prediction time per row of the top 3 models in each run.

In [17]:
print(f"Total run time, all features: {aml.total_secs:.1f} s (H2O reports {aml.training_info['duration_secs']} s)")
print(f"Total run time, top 3 features: {aml3.total_secs:.1f} s (H2O reports {aml3.training_info['duration_secs']} s)")


def speed_table(a):
    board = h2o.automl.get_leaderboard(a, extra_columns="ALL").as_data_frame()
    return board.head(3)[["model_id", "rmse", "training_time_ms", "predict_time_per_row_ms"]]


pd.concat({"All features": speed_table(aml), "Top 3 features": speed_table(aml3)})

Total run time, all features: 110.5 s (H2O reports 56 s)
Total run time, top 3 features: 72.8 s (H2O reports 16 s)


model_id  \
All features   0     StackedEnsemble_AllModels_1_AutoML_6_20261101_14558   
               1  StackedEnsemble_BestOfFamily_1_AutoML_6_20261101_14558   
               2                           GBM_5_AutoML_6_20261101_14558   
Top 3 features 0     StackedEnsemble_AllModels_1_AutoML_7_20261101_14807   
               1  StackedEnsemble_BestOfFamily_1_AutoML_7_20261101_14807   
               2                           GBM_5_AutoML_7_20261101_14807   

                       rmse  training_time_ms  predict_time_per_row_ms  
All features   0  36.601986              1165                 0.016849  
               1  36.633465               443                 0.005876  
               2  37.003890               255                 0.001036  
Top 3 features 0  37.993438              1094                 0.009794  
               1  38.010017               411                 0.003661  
               2  38.052683               134                 0.000857